# Predictive Analytics: Support Vector Machines with Classification

We used the GPU to train this model. In case the model shoulde be trained on the CPU. Change USE_GPU to false.

In [1]:
USE_GPU = True

In [2]:
if USE_GPU:
    %load_ext cuml.accel
from run_config import PATHS

In [3]:
if USE_GPU:
    import os
    os.environ["LD_LIBRARY_PATH"] = "/mnt/c/Users/bkran/Documents/AAA/Group-3-AAA/.venv/lib64/python3.12/site-packages/nvidia/cuda_runtime/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

    import cuml
    print(cuml.__version__)

26.06.00


In [4]:
TRAIN_SAMPLE = 500_000 # gets balanced
GRID_SAMPLE = 60_000 # does not get balanced
SPATIAL_UNIT = "CENSUS_TRACTS" # options: CENSUS_TRACTS, HEXAGON, COMMUNITY_AREAS
SPATIAL_ENCODING = "latlong" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "1H" # options: 1H, 2H, 4H
H3_RES = "8" # options 7,8

In [5]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [6]:
import pandas as pd
import geopandas as gpd
import numpy as np
import polars as pl
from shapely import wkt

if USE_GPU:
    # cuml
    from cuml import SVC
    from cuml import LinearSVC
else:
    #sklearn
    from sklearn.svm import SVR 
    from sklearn.svm import LinearSVR
    
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV 
from sklearn.experimental import enable_halving_search_cv # noqa
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.utils import resample
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_curve, precision_score, classification_report, confusion_matrix

import h3

# joblib
from joblib import load, dump
from joblib import Memory

from imblearn.under_sampling import RandomUnderSampler




## Preparations

In [7]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [8]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TRAIN.parquet"
DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_TEST.parquet"
DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_VAL.parquet"
if SPATIAL_UNIT == "HEXAGON":
    DATA_PATH_TEST = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_TEST.parquet"
    DATA_PATH_TRAIN = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_TRAIN.parquet"
    DATA_PATH_VAL = INPUT + "GOLD_" + TIME_UNIT + "_DEMAND_" + SPATIAL_UNIT + "_" + H3_RES + "_VAL.parquet"



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL, # gets encoded
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
    "h3_cell", # spatial units are getting encoded
    "census_tract",
    "community_area",
    "lat",
    "lon"    
]

Load data and select features and target

In [9]:
# Load data
train_df = pd.read_parquet(DATA_PATH_TRAIN)
test_df = pd.read_parquet(DATA_PATH_TEST)
val_df = pd.read_parquet(DATA_PATH_VAL)

In [10]:
if len(train_df) > TRAIN_SAMPLE:
   train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=42)

In [11]:
def trip_demand(df):
    df["trip_demand"] = 1  # default class for trip_count > 0 but below all thresholds
    df.loc[df["trip_count"] == 0, "trip_demand"] = 0
    df.loc[df["trip_count"] >= 5, "trip_demand"] = 2
    df.loc[df["trip_count"] >= 25, "trip_demand"] = 3
    df.loc[df["trip_count"] >= 100, "trip_demand"] = 4
    return df

train_df = trip_demand(train_df)
val_df = trip_demand(val_df)
test_df = trip_demand(test_df)

In [12]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand
574296,2025-03-02 13:00:00,3,7,13,0.866025,5.000000e-01,-0.781831,0.623490,-0.258819,-0.965926,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
1634966,2025-11-03 02:00:00,11,1,2,-0.866025,5.000000e-01,0.000000,1.000000,0.500000,0.866025,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
4493997,2025-09-04 16:00:00,9,4,16,-0.866025,-5.000000e-01,0.433884,-0.900969,-0.866025,-0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
4723951,2026-01-20 08:00:00,1,2,8,0.000000,1.000000e+00,0.781831,0.623490,0.866025,-0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
3667150,2025-06-05 17:00:00,6,4,17,0.500000,-8.660254e-01,0.433884,-0.900969,-0.965926,-0.258819,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6310273,2025-01-10 08:00:00,1,5,8,0.000000,1.000000e+00,-0.433884,-0.900969,0.866025,-0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
949571,2026-03-22 15:00:00,3,7,15,0.866025,5.000000e-01,-0.781831,0.623490,-0.707107,-0.707107,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
6900162,2025-08-23 13:00:00,8,6,13,-0.500000,-8.660254e-01,-0.974928,-0.222521,-0.258819,-0.965926,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0
6743833,2025-01-19 04:00:00,1,7,4,0.000000,1.000000e+00,-0.781831,0.623490,0.866025,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No trips,0


In [13]:
model = SVC()

## Encoding

In [14]:
def feature_cols(train_df):
    feature_cols = [
        col for col in train_df.columns
        if col not in EXCLUDE_COLS
    ]
    return feature_cols


### Spatial Encoding: LatLong

In [15]:
def spherical_encode(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return np.stack([x, y, z], axis=-1)  

In [16]:
# encode into lat long


if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "HEXAGON":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, val_df, test_df):
            df["lat"], df["lon"] = zip(*df["h3_cell"].map(h3.cell_to_latlng))

    elif SPATIAL_UNIT == "CENSUS_TRACTS":
        print("Encoding: latlong and Unit: census_tract")

        # load census tract
        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, val_df, test_df):
            df["census_tract"] = df["census_tract"].astype(str).str.zfill(11)
            df["lat"] = df["census_tract"].map(tract_centroids["lat"])
            df["lon"] = df["census_tract"].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

    elif SPATIAL_UNIT == "COMMUNITY_AREAS":
        print("Encoding: latlong and Unit: community_area")

        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  


        gdf_proj = gdf.to_crs(epsg=3435)
        gdf["lon"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).x
        gdf["lat"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, val_df, test_df):
            df["community_area"] = df["community_area"].astype(str).str.zfill(2)
            df["lat"] = df["community_area"].map(tract_centroids["lat"])
            df["lon"] = df["community_area"].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")




Encoding: latlong and Unit: census_tract


In [17]:
if SPATIAL_ENCODING == "latlong":
    for df in (train_df, val_df, test_df):
        result = spherical_encode(df["lat"], df["lon"])  
        df["x"], df["y"], df["z"] = result.T  

    # add x, y, z 
    feature_cols = feature_cols(train_df)

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    if len(val_df) > GRID_SAMPLE:
        val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    else: 
        val_df_grid = val_df
    X_val_grid = val_df_grid[feature_cols]

In [18]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand,lat,lon,x,y,z
574296,2025-03-02 13:00:00,3,7,13,0.866025,0.500000,-0.781831,0.623490,-0.258819,-0.965926,...,0.0,0.0,0.0,No trips,0,41.840653,-87.638749,0.030694,-0.744370,0.667061
1634966,2025-11-03 02:00:00,11,1,2,-0.866025,0.500000,0.000000,1.000000,0.500000,0.866025,...,0.0,0.0,0.0,No trips,0,41.805854,-87.608929,0.031098,-0.744759,0.666609
4493997,2025-09-04 16:00:00,9,4,16,-0.866025,-0.500000,0.433884,-0.900969,-0.866025,-0.500000,...,0.0,0.0,0.0,No trips,0,41.927819,-87.751363,0.029191,-0.743414,0.668194
4723951,2026-01-20 08:00:00,1,2,8,0.000000,1.000000,0.781831,0.623490,0.866025,-0.500000,...,0.0,0.0,0.0,No trips,0,41.798212,-87.800555,0.028611,-0.744948,0.666509
3667150,2025-06-05 17:00:00,6,4,17,0.500000,-0.866025,0.433884,-0.900969,-0.965926,-0.258819,...,0.0,0.0,0.0,No trips,0,41.775245,-87.707956,0.029825,-0.745167,0.666210


### Spatial Encoding: OneHotEncoding

In [19]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "COMMUNITY_AREAS"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
   # X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train
    X_test = X_test.reindex(columns=train_columns, fill_value=0)
    #X_val = X_val.reindex(columns=train_columns, fill_value=0)

    val_df_grid = val_df.sample(n=GRID_SAMPLE, random_state=42)
    X_val_grid = pd.get_dummies(val_df_grid[feature_cols], columns=["community_area"])
    X_val_grid = X_val_grid.reindex(columns=train_columns, fill_value=0)


In [20]:
X_train.head()

,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,is_holiday,weather_station_distance_km,food_drink,landmark,...,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,x,y,z
574296,0.866025,0.500000,-0.781831,0.623490,-0.258819,-0.965926,0,11.209340,1.0,1.0,...,0,1,0,0,1,0,0,0.030694,-0.744370,0.667061
1634966,-0.866025,0.500000,0.000000,1.000000,0.500000,0.866025,0,12.096691,6.0,0.0,...,1,0,0,0,1,0,0,0.031098,-0.744759,0.666609
4493997,-0.866025,-0.500000,0.433884,-0.900969,-0.866025,-0.500000,0,15.335514,4.0,0.0,...,0,0,0,0,0,1,0,0.029191,-0.743414,0.668194
4723951,0.000000,1.000000,0.781831,0.623490,0.866025,-0.500000,0,4.216867,0.0,0.0,...,0,0,0,0,1,0,0,0.028611,-0.744948,0.666509
3667150,0.500000,-0.866025,0.433884,-0.900969,-0.965926,-0.258819,0,3.874400,2.0,0.0,...,0,0,1,0,1,0,0,0.029825,-0.745167,0.666210


### Create y

In [21]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_val_grid = val_df_grid[TARGET_COL]

In [22]:
rus = RandomUnderSampler(random_state=0)
X_train, y_train = rus.fit_resample(X_train, y_train)

## Grid Search

In [23]:
memory = Memory(location="/tmp/sklearn_cache", verbose=0)

# pipelines
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(max_iter=100_000, tol=1e-2))
])

pipe_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_map', Nystroem()),
    ('svm', SVC(max_iter=50_000, tol=1e-2))
], memory=memory)

param_grid_linear = {
    "svm__C": [1, 10, 30, 100],
}

param_grid_rbf_sigmoid = {
    "svm__C": [1, 10],
    "feature_map__kernel": ["rbf", "sigmoid"],
    "feature_map__gamma": [0.01, 0.1, 1],
    "feature_map__n_components": [100, 300],
}

param_grid_poly = {
    "svm__C": [1, 10],
    "feature_map__kernel": ["poly"],
    "feature_map__degree": [3, 4],
    "feature_map__gamma": [0.01, 0.1, 1],
    "feature_map__n_components": [100, 300],
}

grids = {}
configs = [
    ("linear", pipe_linear, param_grid_linear),
    ("rbf_sigmoid", pipe_kernel, param_grid_rbf_sigmoid),
    ("poly", pipe_kernel, param_grid_poly),
]

for name, pipe, grid in configs:
    search = HalvingGridSearchCV(
        estimator=pipe,
        param_grid=grid,
        cv=2,
        scoring="balanced_accuracy",
        n_jobs=1,
        error_score="raise"
    )
    search.fit(X_val_grid, y_val_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.2022657858923675 best params: {'svm__C': 30}
rbf_sigmoid best score: 0.42302339754765556 best params: {'feature_map__gamma': 0.01, 'feature_map__kernel': 'rbf', 'feature_map__n_components': 300, 'svm__C': 10}
poly best score: 0.3686440450544226 best params: {'feature_map__degree': 4, 'feature_map__gamma': 0.01, 'feature_map__kernel': 'poly', 'feature_map__n_components': 100, 'svm__C': 10}
Overall best: rbf_sigmoid {'feature_map__gamma': 0.01, 'feature_map__kernel': 'rbf', 'feature_map__n_components': 300, 'svm__C': 10}


In [24]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'feature_map__gamma': 0.01, 'feature_map__kernel': 'rbf', 'feature_map__n_components': 300, 'svm__C': 10}
Best CV score: 0.42302339754765556


In [25]:
# Evaluate on test set
best_model = grid_search.best_estimator_


## Training Model

In [26]:
# Train SVC 
best_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('feature_map', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",Memory(locati..._cache/joblib)
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: str or callable, default='rbf'Kernel map to be approximated. A callable should accept two argumentsand the keyword arguments passed to this object as `kernel_params`, andshould return a floating point number.",'rbf'
,"gamma gamma: float, default=NoneGamma parameter for the RBF, laplacian, polynomial, exponential chi2and sigmoid kernels. Interpretation of the default value is left tothe kernel; see the documentation for sklearn.metrics.pairwise.Ignored by other kernels.",0.01
,"coef0 coef0: float, default=NoneZero coefficient for polynomial and sigmoid kernels.Ignored by other kernels.",None
,"degree degree: float, default=NoneDegree of the polynomial kernel. Ignored by other kernels.",None


## Testing Model

In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[105280  16990    992     88      7]
 [   384   2463    761     60      3]
 [    12    682   2003    524     12]
 [     4     35    393   1632    390]
 [     2      4     27    331   1825]]
              precision    recall  f1-score   support

           0       1.00      0.85      0.92    123357
           1       0.12      0.67      0.21      3671
           2       0.48      0.62      0.54      3233
           3       0.62      0.67      0.64      2454
           4       0.82      0.83      0.82      2189

    accuracy                           0.84    134904
   macro avg       0.61      0.73      0.63    134904
weighted avg       0.95      0.84      0.88    134904



## Save Model and Grid Search

In [ ]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")
dump(grid_search, "../models/grid_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")

['../models/grid_COMMUNITY_AREAS_1H_svc_latlong.joblib']

In [ ]:

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
                      index=[f"Actual_{c}" for c in sorted(set(y_test))],
                      columns=[f"Predicted_{c}" for c in sorted(set(y_test))])
cm_df.to_csv("confusion_matrix.csv")


report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("../models/svm/class_report_" + SPATIAL_UNIT + "_" + H3_RES + "_" + TIME_UNIT + ".csv", index=False)

[[105280  16990    992     88      7]
 [   384   2463    761     60      3]
 [    12    682   2003    524     12]
 [     4     35    393   1632    390]
 [     2      4     27    331   1825]]
              precision    recall  f1-score   support

           0       1.00      0.85      0.92    123357
           1       0.12      0.67      0.21      3671
           2       0.48      0.62      0.54      3233
           3       0.62      0.67      0.64      2454
           4       0.82      0.83      0.82      2189

    accuracy                           0.84    134904
   macro avg       0.61      0.73      0.63    134904
weighted avg       0.95      0.84      0.88    134904



In [ ]:
if SPATIAL_UNIT == "HEXAGON": 
    df = pd.DataFrame({ # did not reorder at any point
        "y_pred": y_pred,
        "y_test": y_test,
        "h3_cell": test_df["h3_cell"].values,
        "date": test_df["datetime_hour"].values,
    })
    df.to_csv("../models/svm/test/svc_" + SPATIAL_UNIT + "_" + H3_RES + "_" + TIME_UNIT + ".csv", index=False)
elif SPATIAL_UNIT == "CENSUS_TRACTS":
    df = pd.DataFrame({ # did not reorder at any point
        "y_pred": y_pred,
        "y_test": y_test,
        "census_tract": test_df["census_tract"].values,
        "date": test_df["datetime_hour"].values,
    })
    df.to_csv("../models/svm/svc_" + SPATIAL_UNIT + "_" + TIME_UNIT + ".csv", index=False)
elif SPATIAL_UNIT == "COMMUNITY_AREAS":
    df = pd.DataFrame({ # did not reorder at any point
        "y_pred": y_pred,
        "y_test": y_test,
        "community_area": test_df["community_area"].values,
        "date": test_df["datetime_hour"].values,
    })
    df.to_csv("../models/svm/svc_" + SPATIAL_UNIT + "_" + TIME_UNIT + ".csv", index=False)
else: 
    print("No file was created.")
